<a href="https://colab.research.google.com/github/Nazeem0/CODEDEX/blob/main/Copy_of_ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
from google.colab import userdata

PROJECT_ID = userdata.get("PROJECT_ID")
LOCATION = userdata.get("LOCATION")

print(PROJECT_ID)
print(LOCATION)

48888695514
Singapore


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [2]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [6]:
!gcloud config set project {PROJECT_ID}

Are you sure you wish to set property [core/project] to PROJECT_ID?

Do you want to continue (Y/n)?  y

ERROR: (gcloud.config.set) The project property must be set to a valid project ID, not the project name [PROJECT_ID]
To set your project, run:

  $ gcloud config set project PROJECT_ID

or to unset it, run:

  $ gcloud config unset project


In [13]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "project-nazeem"             # @param {type:"string"}
LOCATION = "asia-southeast1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'project-nazeem' in 'asia-southeast1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [14]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [15]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [16]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: 'c1ac356c-8d6a-424c-93e2-5b377c24eebe'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable!

---

## Relaxing & Artsy Day Trip: Stanford & Palo Alto

This itinerary focuses on the beautiful Stanford University campus and the charming city of Palo Alto, offering a blend of world-class art, serene outdoor spaces, and budget-friendly food options.

### Morning (10:00 AM - 1:00 PM): Artistic Immersion at Stanford

*   **10:00 AM - 12:30 PM: Cantor Arts Center & Rodin Sculpture Garden.**
    Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission is always free, making it a perfect affordable choice for art lovers. The museum is open on Sundays from 10:00 AM to 5:00 PM. It hou

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable!

---

## Relaxing & Artsy Day Trip: Stanford & Palo Alto

This itinerary focuses on the beautiful Stanford University campus and the charming city of Palo Alto, offering a blend of world-class art, serene outdoor spaces, and budget-friendly food options.

### Morning (10:00 AM - 1:00 PM): Artistic Immersion at Stanford

*   **10:00 AM - 12:30 PM: Cantor Arts Center & Rodin Sculpture Garden.**
    Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission is always free, making it a perfect affordable choice for art lovers. The museum is open on Sundays from 10:00 AM to 5:00 PM. It houses a diverse collection spanning 5,000 years of art. Don't miss the iconic **Rodin Sculpture Garden**, located just outside the museum. This outdoor gallery is free and open 24/7, featuring the largest collection of Auguste Rodin's bronze sculptures outside of Paris, including "The Thinker" and "The Gates of Hell." Stroll through the tranquil garden and admire the powerful works.
*   **12:30 PM - 1:00 PM: Explore Stanford University Architecture.**
    After the museum, take a leisurely walk through the stunning Stanford University campus. Admire the distinctive Romanesque Revival architecture, often referred to as "Leland Stanford Junior University" style. The vast, beautiful grounds offer a relaxing atmosphere for a gentle stroll.

### Lunch (1:00 PM - 2:30 PM): Affordable Bites in Palo Alto

Head into downtown Palo Alto for a budget-friendly lunch.
*   **Recommendation:**
    *   **Mediterranean Wraps:** Known for its flavorful and affordable Mediterranean cuisine, especially their highly-regarded falafel.
    *   **Sancho's Taqueria:** Offers authentic Mexican food in a lively atmosphere, known for its fish tacos, but remember it's cash-only.
    *   **Spice Kit:** For Asian street food with a "Chipotle-style" service, allowing you to customize your meal affordably.

### Afternoon (2:30 PM - 5:30 PM): Contemporary Art & Campus Serenity

*   **2:30 PM - 4:00 PM: Anderson Collection at Stanford University.**
    Walk over to the **Anderson Collection**, also on the Stanford campus and offering free admission. This museum focuses on modern and contemporary American art, showcasing works by renowned artists such as Jackson Pollock and Wayne Thiebaud. Enjoy a more intimate art experience here.
*   **4:00 PM - 5:30 PM: Relax at the Arizona Garden.**
    For a unique and relaxing experience, visit Stanford's **Arizona Garden**. This historic succulent garden is a peaceful oasis perfect for unwinding amidst fascinating desert plants and intricate landscaping. It's a quiet spot to sit, reflect, and enjoy nature.

### Evening (5:30 PM - 7:30 PM): Sunset Views & Simple Dinner

*   **5:30 PM - 7:00 PM: Sunset at Palo Alto Baylands Nature Preserve.**
    Drive a short distance to the **Palo Alto Baylands Nature Preserve**. This expansive preserve offers flat trails and stunning views over wetlands and salt marshes, providing a fantastic opportunity to watch the sunset over the Bay. It's a completely free and relaxing way to end your day, offering wide-open spaces and minimal light pollution for optimal viewing. Alternatively, **Pearson-Arastradero Preserve** also offers beautiful sunset views over rolling hills.
*   **7:00 PM onwards: Casual & Affordable Dinner.**
    For a final affordable meal, consider picking up takeout from one of Palo Alto's many casual eateries, or revisit one of the affordable lunch spots you found earlier. Enjoy your meal under the stars or on your drive back to Sunnyvale.

---

**Important Notes:**
*   **Parking:** While parking at the Cantor Arts Center is generally free on weekends, please be aware that the search results indicate "BTS is visiting campus this weekend! Traffic and parking will be impacted May 16 and 17, with potential impacts on May 15 and 18 as well." It's advisable to check Stanford's transportation website on the day of your visit for real-time parking availability and consider carpooling or arriving earlier if possible.
*   **Operating Hours:** All listed museum hours are for today, Sunday, May 17, 2026. However, always check the official websites for the most up-to-date information before your trip, as hours can change for special events or holidays.

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [17]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [18]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: 'e1a2540b-925e-4275-bbe6-5b97a54c2115'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-a1df8f4b-fac4-4351-93b8-3ec49d4f2f63',
        name='get_live_weather_forecast'
      ),
      thought_signature=b"\n\x86\x02\x01\x8f=k_\x93\x1a\x93\xdd\x1e\x87j\xb1\xd1'\xe4\x8awdY\xf4p\x8f3\xf2$\xc8My\x01\xae\x9cS<\xb9\xf2\xfa\xc4Q\xfe\x0c+\xe1:j\x11\x88\x97W\xb9A\x96\x1e\xd5\xf2u\xeca\x157{\xf2\xb2FJ\xba\xc8t\xc2\x91\x087\xbf`\x84FX\xbf^\x83\x80\xce;J\xa7^\x8a\xb7i&\xabG/\xd7...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
    ),
  ],
  prompt_tok

The weather near Lake Tahoe is mostly sunny with a high of 51°F. There will be a north wind of 5 to 20 mph, with gusts as high as 40 mph. It might be a bit windy for hiking, so make sure to dress in layers and be prepared for strong gusts.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [19]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [20]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: 'cb5da4c8-ede5-46e8-80de-9860b9c0c8c1'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Find the top-rated hotels in San Francisco'
        },
        id='adk-5b050099-e5af-4a3d-901b-fcbbefbbda9c',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x9e\x03\x01\x8f=k_Bn\xdfTG\xab\x97\xd6\x18\x83\x8f\xdbP\x0c\xe8PrU\xbd\x85\xe5\xac\x9d\xaeVR\xed\xb2\x89\xb1\xc3\xea\xb5\xca\xec\xff\xf9\x8fu\xbd~=n\xe5\xa9\x01C\xcc\xd6\xf8\x9a/\xe8<K\xe3>\xc47\xda\xb3\xc2\x19\x87l\xb1\xbb\xf5dzP4\x1d\x17\x1a\xd2\x15$\xe4\tQ\x91\xf6\xa4\xab\x1e-9\xc5...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=

Certainly! Based on our culinary expert's opinion, for an exceptional dining experience, I recommend **State Bird Provisions**. Our expert describes it as a place where your "palate demands actual culinary genius over tourist traps." I trust you'll find it delightful. Enjoy your dinner!

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [21]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [22]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 14ff0420-2a9f-4bbd-82e3-29316ccc800a

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '14ff0420-2a9f-4bbd-82e3-29316ccc800a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! A 2-day trip to Lisbon sounds fantastic, especially with your interests in historic sites and local food. I'll help you plan a memorable itinerary, one day at a time.

Here's a suggestion for your first day in Lisbon:

## Day 1: Historic Alfama & Downtown Delights

**Morning (9:00 AM - 1:00 PM): Explore Alfama's Charms**
Begin your day by immersing yourself in **Alfama**, Lisbon's oldest district, known for its narrow, winding streets and rich history.

*   **São Jorge Castle (Castelo de São Jorge):** Start your exploration at this ico

Great! A 2-day trip to Lisbon sounds fantastic, especially with your interests in historic sites and local food. I'll help you plan a memorable itinerary, one day at a time.

Here's a suggestion for your first day in Lisbon:

## Day 1: Historic Alfama & Downtown Delights

**Morning (9:00 AM - 1:00 PM): Explore Alfama's Charms**
Begin your day by immersing yourself in **Alfama**, Lisbon's oldest district, known for its narrow, winding streets and rich history.

*   **São Jorge Castle (Castelo de São Jorge):** Start your exploration at this iconic medieval castle perched atop one of Lisbon's hills. You'll get stunning panoramic views of the city and the Tagus River. Take your time to walk through the fortress, its gardens, and the archaeological site.
*   **Wander Alfama's Streets:** After the castle, get lost (in a good way!) in the labyrinthine alleys of Alfama. This historic neighborhood is a testament to Lisbon's Moorish past.
*   **Lisbon Cathedral (Sé de Lisboa):** Make your way down to the Lisbon Cathedral, a Romanesque structure dating back to the 12th century. It has withstood numerous earthquakes and features impressive Gothic cloisters.

**Lunch (1:30 PM - 2:30 PM): Culinary Journey at Time Out Market**
*   Head to the **Time Out Market (Mercado da Ribeira)**. This bustling food hall offers a curated selection of Lisbon's best culinary offerings under one roof, perfect for sampling various traditional Portuguese dishes.

**Afternoon (2:30 PM - 6:00 PM): Baixa District & Scenic Ride**
*   **Baixa District:** Stroll through the elegant **Baixa** district, rebuilt after the 1755 earthquake with a grid-like street plan. Explore **Rua Augusta**, a pedestrian street known for shopping, and admire the majestic **Praça do Comércio** (Commerce Square) by the Tagus River.
*   **Tram 28 Ride (Optional):** For a quintessential Lisbon experience, consider hopping on the iconic **Tram 28**. This historic yellow tram weaves through some of Lisbon's most picturesque neighborhoods, including Alfama and Baixa, offering unique perspectives of the city.

**Dinner (7:30 PM onwards): Authentic Portuguese Flavors**
*   For dinner, I recommend trying a local restaurant in one of the historic areas. Consider **O Velho Eurico**, a highly-rated hidden gem near São Jorge Castle, known for its fresh and delicious Portuguese dishes like octopus and tuna. Alternatively, **Frangasqueira Nacional** offers excellent peri-peri chicken and other grilled meats. If you prefer an establishment with deep roots, **Café-Restaurante Martinho da Arcada Marquês**, the oldest cafe in Lisbon, also serves traditional Portuguese food.

How does this sound for your first day in Lisbon? Let me know if you'd like any adjustments!

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '14ff0420-2a9f-4bbd-82e3-29316ccc800a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Understood! No problem at all, we can certainly swap out the castle for another historical gem. Lisbon has a wealth of historical sites to offer.

Here's an updated plan for your first day, replacing the castle visit with another historical exploration:

## Day 1: Historic Alfama & Downtown Delights (Revised Morning)

**Morning (9:00 AM - 1:00 PM): Explore Alfama's Charms & Ancient History**
Begin your day by immersing yourself in **Alfama**, Lisbon's oldest district, known for its narrow, winding streets and rich history.

*   **Lisbon Cathedral (Sé de Lisboa):** Start your

Understood! No problem at all, we can certainly swap out the castle for another historical gem. Lisbon has a wealth of historical sites to offer.

Here's an updated plan for your first day, replacing the castle visit with another historical exploration:

## Day 1: Historic Alfama & Downtown Delights (Revised Morning)

**Morning (9:00 AM - 1:00 PM): Explore Alfama's Charms & Ancient History**
Begin your day by immersing yourself in **Alfama**, Lisbon's oldest district, known for its narrow, winding streets and rich history.

*   **Lisbon Cathedral (Sé de Lisboa):** Start your exploration at the Lisbon Cathedral, a magnificent Romanesque structure dating back to the 12th century. It has withstood numerous earthquakes and features impressive Gothic cloisters you can explore.
*   **Roman Theater Museum (Teatro Romano):** Just a short walk from the Cathedral, delve deeper into Lisbon's ancient past at the Roman Theater Museum. You can see the ruins of the ancient Roman theatre, which dates back to the 1st century AD, and learn about the city's Roman heritage.
*   **Wander Alfama's Streets:** After the museum, get lost (in a good way!) in the labyrinthine alleys of Alfama. This historic neighborhood is a testament to Lisbon's Moorish past, with hidden viewpoints, traditional Fado houses, and charming squares.

**Lunch (1:30 PM - 2:30 PM): Culinary Journey at Time Out Market**
*   Head to the **Time Out Market (Mercado da Ribeira)**. This bustling food hall offers a curated selection of Lisbon's best culinary offerings under one roof, perfect for sampling various traditional Portuguese dishes.

**Afternoon (2:30 PM - 6:00 PM): Baixa District & Scenic Ride**
*   **Baixa District:** Stroll through the elegant **Baixa** district, rebuilt after the 1755 earthquake with a grid-like street plan. Explore **Rua Augusta**, a pedestrian street known for shopping, and admire the majestic **Praça do Comércio** (Commerce Square) by the Tagus River.
*   **Tram 28 Ride (Optional):** For a quintessential Lisbon experience, consider hopping on the iconic **Tram 28**. This historic yellow tram weaves through some of Lisbon's most picturesque neighborhoods, including Alfama and Baixa, offering unique perspectives of the city.

**Dinner (7:30 PM onwards): Authentic Portuguese Flavors**
*   For dinner, I recommend trying a local restaurant in one of the historic areas. Consider **O Velho Eurico**, a highly-rated hidden gem near the Alfama area, known for its fresh and delicious Portuguese dishes like octopus and tuna. Alternatively, **Frangasqueira Nacional** offers excellent peri-peri chicken and other grilled meats. If you prefer an establishment with deep roots, **Café-Restaurante Martinho da Arcada Marquês**, the oldest cafe in Lisbon, also serves traditional Portuguese food.

How does this revised Day 1 plan look to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '14ff0420-2a9f-4bbd-82e3-29316ccc800a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is to your liking. Let's move on to planning your second day in Lisbon, ensuring we weave in more historical sites and delicious local food experiences.

## Day 2: Maritime Discoveries, Iconic Treats & Cultural Evening

**Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries & Iconic Pastries**
Today, we'll venture to the historic district of Belém, a place intrinsically linked to Portugal's Age of Discoveries.

*   **Jerónimos Monastery (Mosteiro dos Jerónimos):** Begin your day at this magnificent UNESCO World Heritage site. This stunning monastery, built in the Manueline style, comme

Excellent! I'm glad Day 1 is to your liking. Let's move on to planning your second day in Lisbon, ensuring we weave in more historical sites and delicious local food experiences.

## Day 2: Maritime Discoveries, Iconic Treats & Cultural Evening

**Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries & Iconic Pastries**
Today, we'll venture to the historic district of Belém, a place intrinsically linked to Portugal's Age of Discoveries.

*   **Jerónimos Monastery (Mosteiro dos Jerónimos):** Begin your day at this magnificent UNESCO World Heritage site. This stunning monastery, built in the Manueline style, commemorates Vasco da Gama's voyage to India and houses his tomb. Allow ample time to admire its intricate carvings and visit the church and cloisters.
*   **Belém Tower (Torre de Belém):** A short walk along the Tagus River will bring you to the iconic Belém Tower, another UNESCO site. This 16th-century fortress once guarded the entrance to Lisbon's harbor and served as a embarkation point for explorers.
*   **Padrão dos Descobrimentos (Monument to the Discoveries):** Nearby, you'll find this impressive monument celebrating Portugal's explorers and their patrons.
*   **Pastéis de Belém:** No visit to Belém is complete without tasting the world-famous *Pastéis de Belém* at the original factory, Fábrica de Pastéis de Belém. Enjoy these warm, crispy custard tarts, often sprinkled with cinnamon and powdered sugar. This is a quintessential Lisbon food experience!

**Lunch (1:30 PM - 2:30 PM): Seafood Delights in Belém**
*   For lunch, enjoy fresh seafood in Belém. There are several excellent restaurants in the area. Consider **Restaurante O Frade** for traditional Portuguese cuisine with a focus on quality ingredients, or a more casual spot along the waterfront.

**Afternoon (2:30 PM - 6:00 PM): Artistic Tiles & Panoramic Views**
*   **National Azulejo Museum (Museu Nacional do Azulejo):** Take a short taxi or bus ride to this unique museum, housed in the former Madre de Deus Convent. It showcases the history and art of *azulejos* (ceramic tiles) in Portugal, from the 15th century to the present day. It's a beautiful and historically rich cultural experience.
*   **Miradouro da Senhora do Monte:** Head to one of Lisbon's most spectacular viewpoints for breathtaking panoramic views of the city as the afternoon light begins to soften. It's a perfect spot for photos and to reflect on your journey.

**Dinner (7:30 PM onwards): Traditional Fado & Portuguese Gastronomy**
*   For your final evening, immerse yourself in a traditional **Fado show with dinner**. Fado, Portugal's soulful and melancholic music, is a UNESCO Intangible Cultural Heritage. Many restaurants in the Alfama or Bairro Alto districts offer a dinner and Fado experience, combining excellent Portuguese cuisine with an unforgettable cultural performance. Consider places like **Clube de Fado** or **Casa de Linhares** for an authentic and high-quality experience. Be sure to book in advance, as these are very popular.

How does this plan for your second day in Lisbon sound?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [23]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 73ddd285-e46d-443a-a90f-7fb4ba41bba1
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '73ddd285-e46d-443a-a90f-7fb4ba41bba1'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! A 2-day trip to Lisbon with a focus on historic sites and great local food sounds wonderful.

Let's start with **Day 1**. How about this plan?

### Day 1: Alfama's Charms and Castles

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & São Jorge Castle**
    Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Soak in the historic atmosphere, discover hidden v

Okay, fantastic! A 2-day trip to Lisbon with a focus on historic sites and great local food sounds wonderful.

Let's start with **Day 1**. How about this plan?

### Day 1: Alfama's Charms and Castles

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & São Jorge Castle**
    Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Soak in the historic atmosphere, discover hidden viewpoints, and then make your way up to São Jorge Castle for panoramic views of the city and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama. Look for dishes like grilled sardines (sardinhas assadas) or bacalhau (codfish) prepared in various ways.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Miradouros**
    Visit the Lisbon Cathedral (Sé de Lisboa), a national monument with a rich history. Afterwards, explore some of Alfama's famous *miradouros* (viewpoints) like Miradouro das Portas do Sol or Miradouro de Santa Luzia for more stunning vistas.
*   **Evening (7:30 PM onwards): Fado & Dinner in Alfama**
    Experience an authentic Fado show, Portugal's soulful music, accompanied by a delicious dinner in one of Alfama's historic Fado houses.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 7303b5f6-65b5-4567-ba1f-67a03f9f513c
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '7303b5f6-65b5-4567-ba1f-67a03f9f513c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here is a plan for Day 2 of your London adventure:

### **Day 2: Royal Traditions & Historic Landmarks**

*   **Morning (9:00 AM - 12:30 PM): Witness the Changing of the Guard at Buckingham Palace**
    Start your day by heading to Buckingham Palace, the official residence of the King. Arrive early to secure a good viewing spot for the iconic Changing of the Guard ceremony, which is scheduled for 11:00 AM on Monday, May 18th. This traditional ceremony, lasting approximately 45 minutes, is a vibrant display of British pageantry and precision. Afterward, take a leisurely stroll through the beautiful St. J

Here is a plan for Day 2 of your London adventure:

### **Day 2: Royal Traditions & Historic Landmarks**

*   **Morning (9:00 AM - 12:30 PM): Witness the Changing of the Guard at Buckingham Palace**
    Start your day by heading to Buckingham Palace, the official residence of the King. Arrive early to secure a good viewing spot for the iconic Changing of the Guard ceremony, which is scheduled for 11:00 AM on Monday, May 18th. This traditional ceremony, lasting approximately 45 minutes, is a vibrant display of British pageantry and precision. Afterward, take a leisurely stroll through the beautiful St. James's Park, located just adjacent to the palace.
*   **Lunch (12:30 PM - 2:00 PM): Casual Bite near St. James's Park**
    Grab a relaxed lunch at one of the many cafes or pubs in the vicinity of Buckingham Palace or St. James's Park.
*   **Afternoon (2:00 PM - 5:00 PM): Explore Westminster Abbey & Houses of Parliament**
    Next, immerse yourself in history with a visit to Westminster Abbey, a magnificent Gothic church that has been the site of coronations, royal weddings, and burials for centuries. The Abbey is typically open until 3:30 PM on Mondays, with the last admission at 2:30 PM, allowing for a good two-hour exploration. Afterward, take a walk past the iconic Houses of Parliament and the Elizabeth Tower, home to Big Ben, for some memorable photo opportunities.
*   **Evening (5:00 PM onwards): Dinner and West End Charm**
    Conclude your day with dinner in London's vibrant West End. Areas like Covent Garden or Soho offer a fantastic array of restaurants, from diverse international cuisines to traditional British fare, and a lively atmosphere for an enjoyable evening.

How does this plan for Day 2 sound to you?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
